# The Odds API - Historical Data Coverage Testing

This notebook systematically tests:
1. How far back historical data goes
2. Player prop availability (points, rebounds, assists)
3. Traditional markets (moneyline, spreads, totals)
4. Date range for player props specifically
5. Bookmaker coverage over time

**CRITICAL INFO FROM DOCS:**
- Historical odds available from June 6, 2020
- 10-minute intervals until September 2022, then 5-minute intervals
- Player props and additional markets available after **May 3, 2023** (2023-05-03T05:30:00Z)
- Cost: 10 credits per market per region for historical data
- 100K credits/month = 10,000 historical queries (1 market, 1 region)

In [2]:
import os
import time
from datetime import datetime

import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("ODDS_API_KEY")
BASE_URL = "https://api.the-odds-api.com/v4"


def track_usage(response):
    """Print usage stats from response headers"""
    remaining = response.headers.get("x-requests-remaining", "N/A")
    used = response.headers.get("x-requests-used", "N/A")
    last = response.headers.get("x-requests-last", "N/A")

    print("\n📊 API USAGE:")
    print(f"   Last call cost: {last}")
    print(f"   Used this month: {used}")
    print(f"   Remaining: {remaining}")
    return int(last) if last != "N/A" else 0

## 1. Test Historical Moneyline Data - How Far Back?

Let's start by testing traditional markets (moneyline) at different historical dates to see what's available.

In [3]:
def test_historical_date(date_str, sport="basketball_nba", markets="h2h", regions="us"):
    """
    Test if historical odds are available for a given date.
    Returns the response data and metadata.
    """
    url = f"{BASE_URL}/historical/sports/{sport}/odds"

    params = {
        "apiKey": API_KEY,
        "regions": regions,
        "markets": markets,
        "date": date_str,
        "oddsFormat": "american",
    }

    print(f"\n🔍 Testing: {date_str}")
    print(f"   Sport: {sport}, Markets: {markets}, Regions: {regions}")

    response = requests.get(url, params=params)
    track_usage(response)

    if response.status_code == 200:
        data = response.json()

        print("\n✅ SUCCESS")
        print(f"   Snapshot timestamp: {data.get('timestamp')}")
        print(f"   Previous available: {data.get('previous_timestamp')}")
        print(f"   Next available: {data.get('next_timestamp')}")
        print(f"   Number of games: {len(data.get('data', []))}")

        if len(data.get("data", [])) > 0:
            game = data["data"][0]
            print("\n   Sample game:")
            print(f"   {game['away_team']} @ {game['home_team']}")
            print(f"   Game time: {game['commence_time']}")
            print(f"   Bookmakers available: {len(game.get('bookmakers', []))}")

            if len(game.get("bookmakers", [])) > 0:
                bookmaker_keys = [bm["key"] for bm in game["bookmakers"]]
                print(f"   Bookmakers: {', '.join(bookmaker_keys[:5])}...")

        return data
    else:
        print(f"\n❌ ERROR: {response.status_code}")
        print(f"   {response.text}")
        return None


# Test dates going backwards
test_dates = [
    "2024-11-15T20:00:00Z",  # Recent
    "2024-01-15T20:00:00Z",  # Earlier 2024
    "2023-01-15T20:00:00Z",  # 2023
    "2022-01-15T20:00:00Z",  # 2022
    "2021-01-15T20:00:00Z",  # 2021
    "2020-06-10T20:00:00Z",  # Near the start (June 6, 2020 is the cutoff)
]

print("=" * 70)
print("TESTING HISTORICAL MONEYLINE AVAILABILITY")
print("=" * 70)

for date in test_dates:
    result = test_historical_date(date)
    time.sleep(1)  # Be nice to the API

TESTING HISTORICAL MONEYLINE AVAILABILITY

🔍 Testing: 2024-11-15T20:00:00Z
   Sport: basketball_nba, Markets: h2h, Regions: us

📊 API USAGE:
   Last call cost: 10
   Used this month: 14
   Remaining: 99986

✅ SUCCESS
   Snapshot timestamp: 2024-11-15T19:55:38Z
   Previous available: 2024-11-15T19:50:38Z
   Next available: 2024-11-15T20:00:38Z
   Number of games: 17

   Sample game:
   Detroit Pistons @ Toronto Raptors
   Game time: 2024-11-16T00:10:00Z
   Bookmakers available: 10
   Bookmakers: draftkings, bovada, mybookieag, fanduel, betrivers...

🔍 Testing: 2024-01-15T20:00:00Z
   Sport: basketball_nba, Markets: h2h, Regions: us

📊 API USAGE:
   Last call cost: 10
   Used this month: 24
   Remaining: 99976

✅ SUCCESS
   Snapshot timestamp: 2024-01-15T19:55:39Z
   Previous available: 2024-01-15T19:50:39Z
   Next available: 2024-01-15T20:00:39Z
   Number of games: 11

   Sample game:
   Houston Rockets @ Philadelphia 76ers
   Game time: 2024-01-15T18:10:54Z
   Bookmakers available: 3
 

## 2. Test Player Props Availability

**CRITICAL**: Player props are only available after May 3, 2023 (2023-05-03T05:30:00Z)

Let's test player props at different dates after this cutoff.

In [4]:
def test_player_props(date_str, prop_types=["player_points", "player_rebounds", "player_assists"]):
    """
    Test player prop availability for multiple prop types.
    Uses historical events endpoint first to get event IDs, then queries props.
    """
    # First, get available events for this date
    events_url = f"{BASE_URL}/historical/sports/basketball_nba/events"
    events_params = {"apiKey": API_KEY, "date": date_str}

    print(f"\n🏀 Testing Player Props: {date_str}")
    print("   Step 1: Getting available events...")

    events_response = requests.get(events_url, params=events_params)
    track_usage(events_response)

    if events_response.status_code != 200:
        print(f"   ❌ Failed to get events: {events_response.status_code}")
        return None

    events_data = events_response.json()
    events = events_data.get("data", [])

    if len(events) == 0:
        print("   ⚠️  No events found for this date")
        return None

    print(f"   Found {len(events)} events")

    # Pick first event to test props
    test_event = events[0]
    event_id = test_event["id"]
    print(f"\n   Step 2: Testing props for event: {test_event['away_team']} @ {test_event['home_team']}")
    print(f"   Event ID: {event_id}")

    # Now test each prop type
    results = {}

    for prop_type in prop_types:
        print(f"\n   Testing {prop_type}...")

        odds_url = f"{BASE_URL}/historical/sports/basketball_nba/events/{event_id}/odds"
        odds_params = {
            "apiKey": API_KEY,
            "date": date_str,
            "regions": "us",
            "markets": prop_type,
            "oddsFormat": "american",
        }

        odds_response = requests.get(odds_url, params=odds_params)
        track_usage(odds_response)

        if odds_response.status_code == 200:
            odds_data = odds_response.json()
            event_data = odds_data.get("data", {})
            bookmakers = event_data.get("bookmakers", [])

            if len(bookmakers) > 0:
                # Count total props available
                total_props = 0
                unique_players = set()
                bookmaker_keys = []

                for bm in bookmakers:
                    bookmaker_keys.append(bm["key"])
                    for market in bm.get("markets", []):
                        if market["key"] == prop_type:
                            for outcome in market.get("outcomes", []):
                                if "description" in outcome:
                                    unique_players.add(outcome["description"])
                            total_props += len(market.get("outcomes", [])) // 2  # Divide by 2 for over/under

                print("      ✅ Available")
                print(f"         Bookmakers: {len(bookmakers)} ({', '.join(bookmaker_keys[:3])}...)")
                print(f"         Unique players: {len(unique_players)}")
                print(f"         Total props: {total_props}")

                # Show sample props
                if len(unique_players) > 0:
                    sample_players = list(unique_players)[:3]
                    print(f"         Sample players: {', '.join(sample_players)}")

                results[prop_type] = {
                    "available": True,
                    "bookmakers": len(bookmakers),
                    "players": len(unique_players),
                    "props": total_props,
                }
            else:
                print(f"      ❌ No bookmakers offering {prop_type}")
                results[prop_type] = {"available": False}
        else:
            print(f"      ❌ Error: {odds_response.status_code}")
            results[prop_type] = {"available": False, "error": odds_response.status_code}

        time.sleep(0.5)  # Rate limiting

    return results


# Test player props at different dates AFTER May 3, 2023
prop_test_dates = [
    "2024-11-15T20:00:00Z",  # Recent
    "2024-03-15T20:00:00Z",  # Mid 2024
    "2023-11-15T20:00:00Z",  # 2023-24 season
    "2023-05-15T20:00:00Z",  # Shortly after May 3, 2023 cutoff
]

print("\n" + "=" * 70)
print("TESTING PLAYER PROPS AVAILABILITY")
print("=" * 70)
print("\nNote: Player props only available after 2023-05-03T05:30:00Z")

for date in prop_test_dates:
    result = test_player_props(date)
    time.sleep(2)  # Be extra nice since we're making multiple calls per date


TESTING PLAYER PROPS AVAILABILITY

Note: Player props only available after 2023-05-03T05:30:00Z

🏀 Testing Player Props: 2024-11-15T20:00:00Z
   Step 1: Getting available events...

📊 API USAGE:
   Last call cost: 1
   Used this month: 55
   Remaining: 99945
   Found 17 events

   Step 2: Testing props for event: Detroit Pistons @ Toronto Raptors
   Event ID: b545064206b8dded3eacfe94a3486010

   Testing player_points...

📊 API USAGE:
   Last call cost: 10
   Used this month: 65
   Remaining: 99935
      ✅ Available
         Bookmakers: 7 (draftkings, bovada, fanduel...)
         Unique players: 11
         Total props: 130
         Sample players: Cade Cunningham, Jalen Duren, Ochai Agbaji

   Testing player_rebounds...

📊 API USAGE:
   Last call cost: 10
   Used this month: 75
   Remaining: 99925
      ✅ Available
         Bookmakers: 7 (draftkings, bovada, fanduel...)
         Unique players: 11
         Total props: 76
         Sample players: Cade Cunningham, Jalen Duren, Ochai Ag

## 3. Test All Traditional Markets

Test availability of spreads, totals, and moneylines for a recent date.

In [5]:
def test_multiple_markets(date_str, markets_list, sport="basketball_nba"):
    """
    Test multiple markets in a single call.

    IMPORTANT: Cost = 10 x [number of markets] x [number of regions]
    This call will cost 10 x len(markets_list) x 1 = 10 * len(markets_list)
    """
    markets_str = ",".join(markets_list)

    url = f"{BASE_URL}/historical/sports/{sport}/odds"
    params = {
        "apiKey": API_KEY,
        "regions": "us",
        "markets": markets_str,
        "date": date_str,
        "oddsFormat": "american",
    }

    print(f"\n🎯 Testing Multiple Markets: {date_str}")
    print(f"   Markets: {markets_str}")
    print(f"   Expected cost: {10 * len(markets_list)} credits")

    response = requests.get(url, params=params)
    track_usage(response)

    if response.status_code == 200:
        data = response.json()
        games = data.get("data", [])

        print(f"\n✅ SUCCESS - Found {len(games)} games")

        if len(games) > 0:
            game = games[0]
            print(f"\n   Sample game: {game['away_team']} @ {game['home_team']}")

            if len(game.get("bookmakers", [])) > 0:
                bm = game["bookmakers"][0]
                print(f"   Bookmaker: {bm['title']}")
                print("   Markets available:")

                for market in bm.get("markets", []):
                    market_key = market["key"]
                    outcomes = market.get("outcomes", [])
                    print(f"      - {market_key}: {len(outcomes)} outcomes")

                    # Show sample outcome
                    if len(outcomes) > 0:
                        outcome = outcomes[0]
                        if "point" in outcome:
                            print(f"        Sample: {outcome['name']} {outcome['point']} ({outcome['price']})")
                        else:
                            print(f"        Sample: {outcome['name']} ({outcome['price']})")

        return data
    else:
        print(f"\n❌ ERROR: {response.status_code}")
        print(f"   {response.text}")
        return None


# Test traditional markets
traditional_markets = ["h2h", "spreads", "totals"]
test_date = "2024-03-15T20:00:00Z"

print("\n" + "=" * 70)
print("TESTING TRADITIONAL MARKETS (MONEYLINE, SPREADS, TOTALS)")
print("=" * 70)

result = test_multiple_markets(test_date, traditional_markets)


TESTING TRADITIONAL MARKETS (MONEYLINE, SPREADS, TOTALS)

🎯 Testing Multiple Markets: 2024-03-15T20:00:00Z
   Markets: h2h,spreads,totals
   Expected cost: 30 credits

📊 API USAGE:
   Last call cost: 30
   Used this month: 208
   Remaining: 99792

✅ SUCCESS - Found 6 games

   Sample game: Phoenix Suns @ Charlotte Hornets
   Bookmaker: BetOnline.ag
   Markets available:
      - h2h: 2 outcomes
        Sample: Charlotte Hornets (385)
      - spreads: 2 outcomes
        Sample: Charlotte Hornets 10.0 (-110)
      - totals: 2 outcomes
        Sample: Over 219.0 (-115)


## 4. Estimate Full Historical Backtest Cost

Calculate the cost to pull all historical player props for your entire date range.

In [6]:
def estimate_backtest_cost():
    """
    Estimate API cost for pulling complete historical data.
    """
    print("\n" + "=" * 70)
    print("COST ESTIMATION FOR FULL HISTORICAL BACKTEST")
    print("=" * 70)

    # Your parameters
    datetime(2023, 5, 4)  # First date with player props
    datetime(2024, 11, 15)  # Recent date

    # NBA schedule

    # Seasons covered
    seasons = [
        ("2023-24 Playoffs", datetime(2023, 5, 4), datetime(2023, 6, 15), 3),
        ("2023-24 Season", datetime(2023, 10, 15), datetime(2024, 4, 15), 8),
        ("2024 Playoffs", datetime(2024, 4, 16), datetime(2024, 6, 20), 3),
        ("2024-25 Season", datetime(2024, 10, 20), datetime(2024, 11, 15), 8),
    ]

    total_cost = 0

    print("\nScenario: Pull player props for ALL games (points, rebounds, assists)")
    print("Cost formula: 10 credits per market per region per API call")
    print("\nApproach 1: Use /historical/sports/{sport}/odds endpoint")
    print("  - Gets all games at once for a given timestamp")
    print("  - Cost: 10 per market (so 30 for 3 prop types)")
    print("  - Need to call multiple times if games spread across day")

    for season_name, start, end, avg_games in seasons:
        days = (end - start).days
        total_games = days * avg_games

        # Assume we need 3 snapshots per day (early, mid, late) to catch all games
        snapshots_per_day = 3
        total_snapshots = days * snapshots_per_day

        # 3 prop types (points, rebounds, assists) = 3 markets
        # Cost per snapshot: 10 x 3 = 30 credits
        cost_per_snapshot = 10 * 3
        season_cost = total_snapshots * cost_per_snapshot

        total_cost += season_cost

        print(f"\n{season_name}:")
        print(f"  Days: {days}")
        print(f"  Est. games: {total_games}")
        print(f"  Snapshots needed: {total_snapshots}")
        print(f"  Cost: {season_cost:,} credits")

    print(f"\n{'=' * 70}")
    print(f"TOTAL ESTIMATED COST: {total_cost:,} credits")
    print("\n100K plan provides: 100,000 credits")
    print(f"This backtest would use: {(total_cost / 100000) * 100:.1f}% of monthly quota")
    print(f"{'=' * 70}")

    print("\n\nApproach 2: Use /historical/sports/{sport}/events/{eventId}/odds")
    print("  - Gets props for a specific game")
    print("  - More precise but needs event IDs first (1 credit each)")
    print("  - Cost: 10 per market per game")

    total_cost_approach2 = 0
    for season_name, start, end, avg_games in seasons:
        days = (end - start).days
        total_games = days * avg_games

        # Need to get events first: days * 1 credit each
        events_cost = days

        # Then get props per game: games * (10 per market * 3 markets)
        props_cost = total_games * (10 * 3)

        season_cost = events_cost + props_cost
        total_cost_approach2 += season_cost

        print(f"\n{season_name}: {season_cost:,} credits")

    print(f"\n{'=' * 70}")
    print(f"TOTAL (Approach 2): {total_cost_approach2:,} credits")
    print(f"This would use: {(total_cost_approach2 / 100000) * 100:.1f}% of monthly quota")
    print(f"{'=' * 70}")

    print("\n\n⚠️  RECOMMENDATION:")
    print("- Approach 1 is more efficient if you want all games for each day")
    print("- Approach 2 is better if you want to be selective about which games to pull")
    print("- For full backtest, Approach 1 costs less if games cluster in time")
    print("- You can do a full historical pull in ONE month with the 100K plan")


estimate_backtest_cost()


COST ESTIMATION FOR FULL HISTORICAL BACKTEST

Scenario: Pull player props for ALL games (points, rebounds, assists)
Cost formula: 10 credits per market per region per API call

Approach 1: Use /historical/sports/{sport}/odds endpoint
  - Gets all games at once for a given timestamp
  - Cost: 10 per market (so 30 for 3 prop types)
  - Need to call multiple times if games spread across day

2023-24 Playoffs:
  Days: 42
  Est. games: 126
  Snapshots needed: 126
  Cost: 3,780 credits

2023-24 Season:
  Days: 183
  Est. games: 1464
  Snapshots needed: 549
  Cost: 16,470 credits

2024 Playoffs:
  Days: 65
  Est. games: 195
  Snapshots needed: 195
  Cost: 5,850 credits

2024-25 Season:
  Days: 26
  Est. games: 208
  Snapshots needed: 78
  Cost: 2,340 credits

TOTAL ESTIMATED COST: 28,440 credits

100K plan provides: 100,000 credits
This backtest would use: 28.4% of monthly quota


Approach 2: Use /historical/sports/{sport}/events/{eventId}/odds
  - Gets props for a specific game
  - More pre

## 5. Summary: What You Need to Know

Run this cell after testing to get a summary report.

In [7]:
print("\n" + "=" * 70)
print("SUMMARY: THE ODDS API FOR NBA PLAYER PROPS")
print("=" * 70)

print("\n📅 HISTORICAL DATA AVAILABILITY:")
print("   Traditional Markets (h2h, spreads, totals):")
print("   ✅ Available from June 6, 2020")
print("   ✅ 10-minute intervals (2020-2022)")
print("   ✅ 5-minute intervals (2022-present)")

print("\n   Player Props (points, rebounds, assists):")
print("   ❌ NOT available before May 3, 2023")
print("   ✅ Available from May 3, 2023 onwards")
print("   ✅ 5-minute intervals")
print("   ⚠️  Coverage varies by bookmaker")

print("\n💰 COST STRUCTURE ($59/month for 100K credits):")
print("   - Current odds: 1 credit per market per region")
print("   - Historical odds: 10 credits per market per region")
print("   - Historical events: 1 credit (to get event IDs)")
print("   - 100K credits = 10,000 historical queries (1 market, 1 region)")

print("\n🎯 FOR YOUR USE CASE (Player Props Prediction):")
print("   Viable date range: May 3, 2023 - Present")
print("   That's ~1.5 NBA seasons worth of data")
print("   Estimated cost for full historical pull: 20K-40K credits")
print("   = Can do it in one month with 100K plan")

print("\n✅ WHAT WORKS WELL:")
print("   - Recent data (2023-present)")
print("   - Multiple bookmakers for line shopping")
print("   - Real-time odds checking during season")
print("   - One-time historical data pull is feasible")

print("\n❌ LIMITATIONS:")
print("   - Can't backtest on pre-2023 player props (data doesn't exist)")
print("   - Limited to ~1.5 seasons of historical props")
print("   - Not all bookmakers offer all prop types")
print("   - Player name matching across books can be inconsistent")

print("\n🔧 RECOMMENDED WORKFLOW:")
print("   1. Sign up for $59/month (100K credits)")
print("   2. Pull historical props from May 2023 - present (one-time)")
print("   3. Store locally in your database")
print("   4. Use remaining credits for real-time odds during season")
print("   5. Compare your predictions to current lines to find edges")

print("\n⚠️  CRITICAL DECISION POINT:")
print("   Can you build a good model with only 1.5 seasons of prop data?")
print("   - Pro: Focus on recent basketball (post-COVID, current rules)")
print("   - Con: Less data for training = higher variance")
print("   - Alternative: Train on game stats (2009-2024), test on props (2023-2024)")

print("\n" + "=" * 70)


SUMMARY: THE ODDS API FOR NBA PLAYER PROPS

📅 HISTORICAL DATA AVAILABILITY:
   Traditional Markets (h2h, spreads, totals):
   ✅ Available from June 6, 2020
   ✅ 10-minute intervals (2020-2022)
   ✅ 5-minute intervals (2022-present)

   Player Props (points, rebounds, assists):
   ❌ NOT available before May 3, 2023
   ✅ Available from May 3, 2023 onwards
   ✅ 5-minute intervals
   ⚠️  Coverage varies by bookmaker

💰 COST STRUCTURE ($59/month for 100K credits):
   - Current odds: 1 credit per market per region
   - Historical odds: 10 credits per market per region
   - Historical events: 1 credit (to get event IDs)
   - 100K credits = 10,000 historical queries (1 market, 1 region)

🎯 FOR YOUR USE CASE (Player Props Prediction):
   Viable date range: May 3, 2023 - Present
   That's ~1.5 NBA seasons worth of data
   Estimated cost for full historical pull: 20K-40K credits
   = Can do it in one month with 100K plan

✅ WHAT WORKS WELL:
   - Recent data (2023-present)
   - Multiple bookmaker